# M3 Agentic AI - Turning functions into tools

## 1. Custom function



## 2. Setup: Initialize environment and client

As in previous labs, you will begin by initializing your environment. You will import several packages now and also later on as you build tools for your LLM.

In [2]:
import sys 
sys.path.append("/Users/maksim/repos/agentic-ai/modules")  
import json
import display_functions
from dotenv import load_dotenv
_ = load_dotenv()

### 2.1 Getting started with AISuite

Next, you'll initialize the **AISuite** client that you've seen in previous lessons. Once initialized, this will be your interface for generating agent responses and calling tools.

Run the cell below to initialize the client.

In [ ]:
import aisuite as ai
import os

# Create an instance of the AISuite client
client = ai.Client()

# Use Openrouter instead of original OpenAU provider
client = ai.Client(provider_configs={"google": {                                                                    
    "base_url": "https://openrouter.ai/api/v1",                                                                     
    "api_key": os.environ["OPENROUTER_API_KEY"]}})    

## 3. Build your first tool

### 3.1 Defining  function



Create function to get time for specific time zone 

In [5]:
import zoneinfo
from datetime import datetime

tz = "Austral"
known = zoneinfo.available_timezones()
suggestions = sorted((item for item in known if (tz or "")[:3].lower() in item.lower()))[:3]
print(f"Suggestions: {list(suggestions)}")

Suggestions: ['America/Manaus', 'Australia/ACT', 'Australia/Adelaide']


In [6]:
# Message structure
prompt = "What time is it in Moscow?"
# prompt = "What time is it in Australia/Sydney?"
messages_tz = [
    {
        "role": "user",
        "content": prompt,
    }
]

In [7]:
import zoneinfo
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
from datetime import datetime

""" 

Lesson:
Catch the bad-zone case and return a JSON-serializable error payload instead of raising. The model         
receives the error message and can self-correct within max_turns — that's the robust-agent pattern this    
module is teaching
"""
def get_current_time_tz(tz:str | None = None) -> tuple[str,str] :
    """
    Returns the current time as a string for the specified IANA time zone.
    Args:
        tz (str | None): Zone name from the zoneinfo database (e.g. "Europe/Paris"). Defaults to "Australia/Sydney" if omitted.
    Returns:
        dict: {"time": "HH:MM:SS", "tz":"<resolved zone>", "error": None} on success, or
        { "time": None, "tz":"<invalid input>", "error": "<message>"} on failure.
    """
    try: 
        zinfo = ZoneInfo(tz) if tz else ZoneInfo("Australia/Sydney")
    except (ZoneInfoNotFoundError, ValueError): 
        known = zoneinfo.available_timezones()
        match = next((k for k in known if k.lower() == (tz or "").lower()), None)
        if match:
            zinfo = ZoneInfo(match)
        else:
            suggestions = sorted((item for item in known if (tz or "")[:3].lower() in item.lower()))[:3]
        return {"time": None, "tz": tz, "error": (
            f"Unknown time zone {tz!r}. RETRY the tool call using exactly one of: " 
            f"Suggestions: {', '.join(suggestions)} or 'Europe/Paris, Australia/Sydney')."
        )}
    return {"time": datetime.now(tz=zinfo).strftime("%H:%M:%S"), "tz": zinfo.key, "error": None}

In [8]:
get_current_time_tz()

{'time': '15:29:56', 'tz': 'Australia/Sydney', 'error': None}

In [9]:
response = client.chat.completions.create(
    model="openrouter:google/gemma-3-27b-it",
    messages=messages_tz,
    tools=[get_current_time_tz],
    max_turns=5
)

# See the LLM response
print(response.choices[0].message.content)

It is 08:30:27 in Moscow.


Great! Just as expected, the function returns a string that has your current time.